# strands-decider playground

A small client for the deployed decider endpoint, plus things to poke at.

> ⚠️ **Experimental.** The endpoint behind this notebook is a single `ml.g6.xlarge` run for
> experimentation, with no SLA and no uptime commitment — it may be shut down at any time.
> Cell 0 checks whether it is still alive. The request/response *contract* is stable; this
> particular deployment is not. See [README.md](../README.md) for the full caveats.

Send a **state** (a ticket, email, document) and a set of **typed questions**; get back
calibrated probabilities. No text generation, no parsing the model's prose.

| question type | you get back |
| --- | --- |
| `noul` | `P(true)` for a yes/no statement |
| `choice` | the chosen option, a probability for every option, and a confidence |
| `score` | an expected level on an ordered rubric, per-level probabilities, a confidence |

See [CLIENT.md](../CLIENT.md) for the full contract and
[ARCHITECTURE.md](../ARCHITECTURE.md) for how the deployment works.

## Before you start

```bash
pip install boto3 notebook
```

You need AWS credentials for the account that hosts the endpoint, with
`sagemaker:InvokeEndpoint` on it. If `aws sts get-caller-identity` works in your shell and
returns the hosting account, this notebook will work too. Nothing below hard-codes an
account id - `boto3` resolves it from your credentials.

There is **no plain URL** to call. This is a SigV4-signed SageMaker endpoint, so
everything goes through `boto3`.

## The client

The only two subtleties, both worth knowing because getting them wrong gives you an
opaque HTTP 500 from Triton's JSON parser:

1. The decider request travels **inside a tensor envelope**, as a JSON **string**.
2. `shape` is `[1, 1]`, not `[1]` - the model sets `max_batch_size > 0`, so Triton
   prepends a batch dimension.

In [ ]:
import json
import statistics
import time
from concurrent.futures import ThreadPoolExecutor

import boto3
from botocore.config import Config
from botocore.exceptions import ClientError

# ---- configuration ---------------------------------------------------------
ENDPOINT = "strands-decider-g6"
REGION = "us-west-2"
# ----------------------------------------------------------------------------


class DeciderError(RuntimeError):
    """The endpoint answered, but with an error body instead of answers.

    `.kind` is 'invalid_request' (your payload), 'internal' (the service), or
    'envelope' (the tensor wrapper was wrong).
    """

    def __init__(self, message, kind):
        super().__init__(f"{kind}: {message}")
        self.kind = kind


# One client, reused. The pool size matters: botocore's default is 10, which silently
# serialises anything above 10 concurrent calls - you would measure the client, not
# the model.
_rt = boto3.client(
    "sagemaker-runtime",
    region_name=REGION,
    config=Config(
        max_pool_connections=64,
        retries={"max_attempts": 2, "mode": "standard"},
        read_timeout=60,  # SageMaker's own hard ceiling for a real-time call
        connect_timeout=10,
    ),
)


def decide(state, questions, endpoint=ENDPOINT):
    """Ask `questions` about `state`. Returns the decider response body.

    `state` may be a string, or any JSON-able dict/list (rendered deterministically
    server-side). Raises DeciderError for a caller or server error.
    """
    inner = json.dumps({"state": state, "questions": questions})
    envelope = {
        "inputs": [{
            "name": "REQUEST_JSON",
            "shape": [1, 1],      # NOT [1]
            "datatype": "BYTES",
            "data": [inner],      # a JSON *string*, not a nested object
        }]
    }

    resp = _rt.invoke_endpoint(
        EndpointName=endpoint,
        ContentType="application/json",
        Body=json.dumps(envelope),
    )
    out = json.loads(resp["Body"].read().decode("utf-8"))

    # A Triton-level failure (bad envelope) has no 'outputs'.
    if "error" in out and "outputs" not in out:
        raise DeciderError(str(out["error"]), "envelope")

    for tensor in out.get("outputs") or []:
        if tensor.get("name") == "RESPONSE_JSON":
            data = tensor.get("data") or []
            if not data:
                raise DeciderError("RESPONSE_JSON carried no data", "internal")
            body = json.loads(data[0])
            # Caller errors arrive as HTTP 200 with an error body, deliberately, so a
            # malformed payload cannot be confused with the model being down.
            if "error" in body:
                err = body["error"]
                raise DeciderError(err.get("message", "unknown"),
                                   err.get("type", "invalid_request"))
            return body

    raise DeciderError(f"no RESPONSE_JSON in {sorted(out)}", "internal")


print(f"client ready -> {ENDPOINT} in {REGION}")

### A readable printer

The raw response is JSON. This renders it so you can see the shape of the distribution,
which is the whole point of a decider model.

In [ ]:
def bar(p, width=30):
    filled = round(max(0.0, min(1.0, p)) * width)
    return "#" * filled + "." * (width - filled)


def show(body, title=None):
    """Print a decider response in a readable form."""
    if title:
        print(f"=== {title}")
    for name, a in body["answers"].items():
        kind = a["type"]
        print(f"\n  {name}  [{kind}]")
        if kind == "noul":
            p = a["noul"]
            verdict = "TRUE" if p >= 0.5 else "false"
            print(f"    P(true) {p:.4f}  {bar(p)}  -> {verdict}")
        elif kind == "choice":
            print(f"    -> {a['choice']}   (confidence {a['confidence']:.3f})")
            ranked = sorted(a["probabilities"].items(), key=lambda kv: -kv[1])
            for opt, p in ranked:
                mark = "  <--" if opt == a["choice"] else ""
                print(f"      {opt:<16} {p:.4f}  {bar(p)}{mark}")
        else:
            print(f"    -> {a['score']:.3f}   (confidence {a['confidence']:.3f})")
            levels = sorted(a["probabilities"].items(), key=lambda kv: int(kv[0]))
            for lvl, p in levels:
                label = a["legend"].get(lvl, "")
                print(f"      {lvl} {label:<14} {p:.4f}  {bar(p)}")
    u = body["usage"]
    print(f"\n  {u['input_tokens']} input tokens, {u['output_tokens']} decision(s), "
          f"{body['latency_ms']} ms server-side\n")


print("printer ready")

## 0. Is it alive?

This is a dev endpoint on one instance and **it may be torn down between uses**, so check
first rather than puzzling over a timeout.

This cell needs `sagemaker:DescribeEndpoint` as well as `InvokeEndpoint`. If you only have
the invoke permission, skip it - the next cell will tell you soon enough.

In [ ]:
sm = boto3.client("sagemaker", region_name=REGION)
try:
    d = sm.describe_endpoint(EndpointName=ENDPOINT)
    v = d["ProductionVariants"][0]
    print(f"{ENDPOINT}: {d['EndpointStatus']}")
    print(f"  variant   {v['VariantName']}")
    print(f"  instances {v['CurrentInstanceCount']} (desired {v['DesiredInstanceCount']})")
    print(f"  config    {d['EndpointConfigName']}")
    if d["EndpointStatus"] != "InService":
        print("\n  NOT InService - invocations will fail.")
        if d.get("FailureReason"):
            print(f"  FailureReason: {d['FailureReason']}")
except ClientError as e:
    code = e.response["Error"]["Code"]
    print(f"could not describe {ENDPOINT}: {code}")
    if code == "ValidationException":
        print("  -> it does not exist. Recreate it: see README.md, 'Deploy'.")
    else:
        print("  -> probably missing sagemaker:DescribeEndpoint. Try the next cell anyway.")

## 1. Your first call

One state, three questions, one of each type. Note that they all come back from a
**single** request - that is the deployment's whole trick.

In [ ]:
body = decide(
    state="Help! My payouts have been failing for 3 days!",
    questions={
        "urgency": {
            "type": "noul",
            "instructions": "Does this convey urgency?",
        },
        "team": {
            "type": "choice",
            "instructions": "Which team should handle this?",
            "criteria": {"billing": None, "sales": None, "retail": None},
        },
        "frustration": {
            "type": "score",
            "instructions": "How frustrated is the writer?",
            "criteria": ["calm", "frustrated", "depressed"],
        },
    },
)

show(body, "failing payouts")

In [ ]:
# The raw body, for when you are wiring this into something.
print(json.dumps(body, indent=2))

## 2. The three question types

### `noul` - yes/no

Returns `P(true)`. **No confidence field**, on purpose: with two outcomes the probability
*is* the uncertainty, so a confidence would just be `|2p - 1|` and tell you nothing new.

Optional `criteria` with `true`/`false` keys sharpens the boundary when the default
reading of your statement is ambiguous.

In [ ]:
STATE = (
    "Hi, my last three payouts have failed and I have had no explanation. "
    "I have contacted support twice already and nobody has got back to me. "
    "This is affecting my ability to pay suppliers. Can someone please call me?"
)

show(decide(STATE, {
    "urgent": {"type": "noul", "instructions": "Does this convey urgency?"},
    "wants_human": {"type": "noul",
                    "instructions": "Is the writer asking to speak to a human?"},
    "wants_refund": {"type": "noul",
                     "instructions": "Is the writer requesting a refund?"},
    # Same question, but with the boundary pinned down:
    "escalate": {
        "type": "noul",
        "instructions": "Should this be escalated to a manager?",
        "criteria": {
            "true": "the customer has been ignored, or money is blocked",
            "false": "a first-line agent can resolve this in one reply",
        },
    },
}), "noul: four yes/no reads on one request")

### `choice` - pick one of N

`criteria` maps option name to a description (or `None` for a bare label). 2 to 255
options. **The option names are yours** - nothing in the model knows what `billing`
means; the prompt alone establishes it, which is why you can invent label sets without
retraining.

`confidence` is normalised max-probability: uniform gives 0.0, one-hot gives 1.0, and it
is independent of the option count, so a threshold means the same thing for a 3-option and
a 10-option question.

**Descriptions matter.** Try running this cell, then delete the descriptions and compare.

In [ ]:
show(decide(STATE, {
    "department": {
        "type": "choice",
        "instructions": "Which department should handle this?",
        "criteria": {
            "billing": "payments, invoices and payouts",
            "technical": "bugs, outages and API errors",
            "shipping": "delivery and tracking",
            "returns": "refunds and exchanges",
            "accounts": "login and profile",
            "sales": "pricing and upgrades",
        },
    },
    "intent": {
        "type": "choice",
        "instructions": "What is the writer's primary intent?",
        # Bare labels - no descriptions.
        "criteria": {k: None for k in
                     ["cancel", "complain", "query", "escalate", "track_order", "other"]},
    },
}), "choice: with descriptions vs bare labels")

### `score` - rate against an ordered rubric

`criteria` is an **ascending** list, 2 to 10 levels; index 0 is the low end.

Two things to internalise:

- `score` is the **expected** level index, not an argmax, so it is fractional. `1.07`
  means "mostly level 1, leaning slightly toward 2".
- `confidence` here is **not** max-probability. It is a normalised standard deviation,
  because on an ordered scale spread is not the same as doubt: mass split between levels
  2 and 3 is confidently saying "about 2.5". Mass at *both ends* is genuinely bad, and
  this formula ranks it far worse than a uniform spread - which max-probability cannot do.

In [ ]:
show(decide(STATE, {
    "frustration": {
        "type": "score",
        "instructions": "How frustrated is the writer?",
        "criteria": ["calm", "frustrated", "depressed"],
    },
    "severity": {
        "type": "score",
        "instructions": "How severe is the business impact?",
        "criteria": ["none", "minor", "moderate", "major", "critical"],
    },
    "politeness": {
        "type": "score",
        "instructions": "How polite is the writer?",
        "criteria": ["abusive", "curt", "neutral", "polite", "warm"],
    },
}), "score: three ordered rubrics")

## 3. Playground - edit this one

Change `MY_STATE` and `MY_QUESTIONS` and re-run. Everything above is just setup.

In [ ]:
MY_STATE = """
Subject: Re: Re: Re: still broken

This is my fourth email. The export job has produced empty files every night
since the 2nd. I have sent you the job IDs twice. We are now a week behind on
our regulatory filing and I have had to explain this to my board.

I would like to know, today, whether this is going to be fixed or whether we
need to start looking at alternatives.
"""

MY_QUESTIONS = {
    "churn_risk": {
        "type": "noul",
        "instructions": "Is this customer at risk of leaving?",
    },
    "owner": {
        "type": "choice",
        "instructions": "Who should own this next?",
        "criteria": {
            "first_line": "a support agent can answer it",
            "engineering": "needs someone who can read logs and ship a fix",
            "account_manager": "the relationship needs attention, not just the bug",
            "legal": "contractual or regulatory exposure",
        },
    },
    "urgency": {
        "type": "score",
        "instructions": "How urgently does this need a reply?",
        "criteria": ["this week", "tomorrow", "today", "within the hour"],
    },
}

show(decide(MY_STATE, MY_QUESTIONS), "playground")

## 4. Ask many questions at once (the thing to get right)

The expensive part of this model is reading the **state**. The engine reads it once and
runs all your questions against that one read, so extra questions are nearly free.

The practical consequence: **never loop one question per request.** This cell measures
the difference on the live endpoint.

In [ ]:
Q7 = {
    "wants_human": {"type": "noul",
                    "instructions": "Is the writer asking to speak to a human?"},
    "urgent": {"type": "noul", "instructions": "Does this convey urgency?"},
    "wants_refund": {"type": "noul",
                     "instructions": "Is the writer requesting a refund?"},
    "department": {"type": "choice",
                   "instructions": "Which department should handle this?",
                   "criteria": {"billing": "payments and invoices",
                                "technical": "bugs and outages",
                                "shipping": "delivery and tracking",
                                "returns": "refunds and exchanges",
                                "accounts": "login and profile",
                                "sales": "pricing and upgrades"}},
    "intent": {"type": "choice", "instructions": "Primary intent?",
               "criteria": {k: None for k in
                            ["cancel", "complain", "query", "escalate", "other"]}},
    "frustration": {"type": "score", "instructions": "How frustrated is the writer?",
                    "criteria": ["calm", "frustrated", "depressed"]},
    "severity": {"type": "score", "instructions": "How severe is the business impact?",
                 "criteria": ["none", "minor", "moderate", "major", "critical"]},
}
Q1 = {"urgent": Q7["urgent"]}


def timed(questions, repeats=5):
    """Median server-side and end-to-end latency, in ms."""
    server, e2e = [], []
    for _ in range(repeats):
        t0 = time.perf_counter()
        out = decide(STATE, questions)
        e2e.append((time.perf_counter() - t0) * 1000)
        server.append(out["latency_ms"])
    return statistics.median(server), statistics.median(e2e)


# Warm both request shapes first: the GPU kernels compile per shape, and a single
# question takes a different code path from several.
decide(STATE, Q1)
decide(STATE, Q7)

s1, e1 = timed(Q1)
s7, e7 = timed(Q7)

print(f"1 question   server {s1:6.1f} ms    end-to-end {e1:7.1f} ms")
print(f"7 questions  server {s7:6.1f} ms    end-to-end {e7:7.1f} ms")
print()
print(f"7x the answers for {s7 / s1:.2f}x the server time")
print(f"per decision: {s1:.1f} ms -> {s7 / 7:.1f} ms")

# Now the wrong way: one call per question.
t0 = time.perf_counter()
for name, q in Q7.items():
    decide(STATE, {name: q})
loop_ms = (time.perf_counter() - t0) * 1000

print()
print(f"7 separate calls      {loop_ms:7.1f} ms")
print(f"1 call, 7 questions   {e7:7.1f} ms")
print(f"-> batching is {loop_ms / e7:.1f}x cheaper for identical answers")

### Long documents are cheap here

Because the state is read once per request rather than once per question, attaching a
document costs you far less than you would expect. Watch `input_tokens` jump while
latency barely moves.

In [ ]:
DOC = ("Transaction log excerpt: payout PO-4471 attempted 2026-09-28 status FAILED "
       "reason INSUFFICIENT_SETTLEMENT_BALANCE; payout PO-4482 attempted 2026-09-30 "
       "status FAILED reason INSUFFICIENT_SETTLEMENT_BALANCE; support case SC-99812 "
       "opened 2026-10-01 severity normal owner unassigned. ")

for label, state in (("plain", STATE),
                     ("with attachment", STATE + " Attached: " + DOC * 4)):
    decide(state, Q7)  # warm this shape
    out = decide(state, Q7)
    print(f"{label:<20} {out['usage']['input_tokens']:>5} tokens   "
          f"{out['latency_ms']:>7.1f} ms server-side")

## 5. Turning confidence into a decision

This is what the probabilities are *for*. Instead of trusting every answer equally, gate
on confidence: act automatically when the model is sure, route to a human when it is not.

A realistic triage policy, run over several requests:

In [ ]:
AUTO_ROUTE_ABOVE = 0.80    # route without review above this choice confidence
ESCALATE_ABOVE = 0.70      # treat a noul as a yes above this

STATES = {
    "clear-cut billing":
        "My invoice shows the wrong tax rate and I need it corrected before I file "
        "my quarterly return. Reference INV-9142.",
    "angry and blocked":
        "Fourth email. Payouts still failing, suppliers unpaid, nobody has replied. "
        "I want someone on the phone today.",
    "genuinely ambiguous":
        "Hi - quick one. Is the thing still happening? Thanks.",
    "happy customer":
        "Just wanted to say the new dashboard is excellent, the export speed is a "
        "huge improvement. No action needed.",
}

TRIAGE = {
    "department": Q7["department"],
    "escalate": {"type": "noul",
                 "instructions": "Should this be escalated to a human immediately?"},
    "severity": Q7["severity"],
}

print(f"{'example':<22} {'route':<12} {'conf':>6}  {'sev':>5}  {'esc':>5}  action")
print("-" * 78)
for label, text in STATES.items():
    a = decide(text, TRIAGE)["answers"]
    dept, conf = a["department"]["choice"], a["department"]["confidence"]
    sev, esc = a["severity"]["score"], a["escalate"]["noul"]

    if esc >= ESCALATE_ABOVE:
        action = "ESCALATE now"
    elif conf >= AUTO_ROUTE_ABOVE:
        action = f"auto-route to {dept}"
    else:
        action = "human triage (model unsure)"

    print(f"{label:<22} {dept:<12} {conf:>6.3f}  {sev:>5.2f}  {esc:>5.3f}  {action}")

## 6. A structured state

`state` does not have to be a string. Pass a dict or list and it is rendered
deterministically server-side (indented JSON, key order preserved), so the same object
always tokenises identically. Handy when the thing you are judging is a record rather than
prose.

In [ ]:
show(decide(
    state={
        "pr": 2291,
        "title": "fix: clamp retry backoff to 30s",
        "files_changed": 3,
        "additions": 18,
        "deletions": 4,
        "tests_added": True,
        "touches": ["src/retry.py", "tests/test_retry.py", "CHANGELOG.md"],
        "ci": "passing",
        "description": "Unbounded backoff meant a flapping dependency could stall "
                       "the worker for hours. Caps it and adds a regression test.",
    },
    questions={
        "safe_to_merge": {"type": "noul",
                          "instructions": "Is this safe to merge without deep review?"},
        "risk": {"type": "score", "instructions": "How risky is this change?",
                 "criteria": ["trivial", "low", "moderate", "high", "dangerous"]},
        "reviewer": {"type": "choice", "instructions": "Who should review it?",
                     "criteria": {"anyone": "a quick second pair of eyes",
                                  "owner": "someone who owns this module",
                                  "architect": "has cross-cutting design implications"}},
    },
), "a pull request as the state")

## 7. Errors

Caller errors come back as **HTTP 200 with an error body**, deliberately: your malformed
payload must not look like the service being down. The client turns those into
`DeciderError`, with `.kind` telling you whose fault it is.

`invalid_request` means fix the payload - retrying will not help. `internal` means retry
with backoff and alert someone.

One thing to expect, because it is confusing the first time: the three question types are
a pydantic **union**, so a validation failure reports *every branch it tried*. A bad
`choice` question produces complaints about `NoulQuestion` as well. The useful line is the
one naming the type you actually sent - the cell below pulls that out for you.

In [ ]:
bad_payloads = [
    ("no questions at all",
     ("x", {})),
    ("choice with only one option",
     ("x", {"q": {"type": "choice", "instructions": "pick",
                  "criteria": {"only": "one"}}})),
    ("score with 11 levels (max is 10)",
     ("x", {"q": {"type": "score", "instructions": "rate",
                  "criteria": [str(i) for i in range(11)]}})),
    ("unknown question type",
     ("x", {"q": {"type": "vibes", "instructions": "how is it"}})),
    ("missing instructions",
     ("x", {"q": {"type": "noul"}})),
]


def relevant(message, sent_type):
    """Pull the useful line out of a pydantic union error.

    The message enumerates every branch of the Question union, so a bad `choice` is
    reported against NoulQuestion, ChoiceQuestion and ScoreQuestion alike. Prefer the
    branch matching the type actually sent; otherwise take the first complaint.
    """
    lines = [ln.strip() for ln in message.splitlines() if ln.strip()]
    lines = [ln for ln in lines if not ln.startswith("For further information")]

    if sent_type:
        want = f"{sent_type.capitalize()}Question"
        for i, ln in enumerate(lines[:-1]):
            if want in ln:
                return f"{ln.rsplit('.', 1)[-1]}: {lines[i + 1]}"

    # No matching branch (e.g. an unrecognised `type`), so show the first complaint.
    for i, ln in enumerate(lines[:-1]):
        if ln.startswith("questions"):
            return f"{ln}: {lines[i + 1]}"
    return lines[-1] if lines else message


for label, (state, questions) in bad_payloads:
    first = next(iter(questions.values()), {}) if questions else {}
    sent_type = first.get("type")
    try:
        decide(state, questions)
        print(f"{label:<32} unexpectedly SUCCEEDED")
    except DeciderError as e:
        print(f"{label:<32} [{e.kind}]")
        print(f"{'':<32} {relevant(str(e), sent_type)[:108]}")

In [ ]:
# One caller's bad payload does not affect requests batched alongside it. Fire a mix
# concurrently and check the good ones all still answer.
def attempt(i):
    try:
        if i % 3 == 0:
            decide("x", {})                       # invalid
            return "bad: unexpectedly ok"
        decide(STATE, Q1)                        # valid
        return "good: answered"
    except DeciderError as e:
        return f"bad: rejected ({e.kind})"


with ThreadPoolExecutor(max_workers=12) as ex:
    results = list(ex.map(attempt, range(12)))

for outcome in sorted(set(results)):
    print(f"{results.count(outcome):>3}x  {outcome}")

## 8. Concurrency - where this deployment earns its keep

Requests that arrive together get coalesced into one GPU pass (up to 8), so throughput
climbs with concurrency even though each individual request gets a little slower.

At concurrency 1 you get none of that benefit, which is the honest caveat about this whole
deployment: its advantage is entirely in what happens when requests overlap.

**Run this from inside `us-west-2` for meaningful numbers.** From a laptop the round trip
dominates and you are measuring the internet, not the GPU - watch the gap between the
`server` and `e2e` columns to see how much.

In [ ]:
SECONDS_PER_CELL = 6.0
CONCURRENCIES = [1, 4, 8]
N_QUESTIONS = len(Q7)

# Distinct requests, so the server cannot collapse them into one shared state - that would
# flatter the result in a way real traffic never does.
POOL = [f"{opener} my {subject} and nobody has replied. Reference {i}. "
        f"This is blocking my work and I need an answer today."
        for i, (opener, subject) in enumerate([
            ("Hi,", "payouts have failed"),
            ("Hello,", "subscription was charged twice"),
            ("Morning,", "order never arrived"),
            ("Hey,", "return label will not generate"),
            ("Dear support,", "login keeps rejecting my password"),
            ("Hi there,", "API keys were revoked without notice"),
            ("Good morning,", "invoice shows the wrong tax rate"),
            ("To whom it may concern,", "refund was approved but never paid"),
        ])]


def drive(start, stride, deadline, server, e2e, errs):
    """One load-generator thread.

    Everything is passed in rather than closed over. The repo's own harnesses
    (tools/sm_sweep.py, tools/bench_tickets.py) make the same choice and say why:
    closing over a sweep loop's variables is the classic way for a harness to
    silently measure the wrong cell once anything becomes concurrent.

    Each thread walks the pool from its own offset, so the requests in flight at any
    instant are different requests rather than the same one N times.
    """
    i = start
    while time.monotonic() < deadline:
        t0 = time.perf_counter()
        try:
            out = decide(POOL[i % len(POOL)], Q7)
            e2e.append((time.perf_counter() - t0) * 1000)
            server.append(out["latency_ms"])
        except Exception as exc:
            # A failed request is a data point, not a reason to abort the sweep.
            errs.append(repr(exc)[:80])
        i += stride


decide(POOL[0], Q7)  # warm the shape

print(f"{'conc':>5} {'tickets':>8} {'requests/s':>10} {'decisions/s':>12} "
      f"{'server p50':>11} {'e2e p50':>9} {'err':>4}")
print("-" * 64)

for conc in CONCURRENCIES:
    server, e2e, errs = [], [], []
    deadline = time.monotonic() + SECONDS_PER_CELL

    with ThreadPoolExecutor(max_workers=conc) as ex:
        futures = [ex.submit(drive, w, conc, deadline, server, e2e, errs)
                   for w in range(conc)]
        for fut in futures:
            fut.result()

    n = len(e2e)
    tps = n / SECONDS_PER_CELL
    s_p50 = statistics.median(server) if server else -1.0
    e_p50 = statistics.median(e2e) if e2e else -1.0
    print(f"{conc:>5} {n:>8} {tps:>10.2f} {tps * N_QUESTIONS:>12.1f} "
          f"{s_p50:>10.1f}m {e_p50:>8.1f}m {len(errs):>4}")
    if errs:
        print(f"      first error: {errs[0]}")

print("\n(reference, measured in-region: ~14 requests/s, ~99 decisions/s at saturation)")

## Notes worth keeping in mind

**Limits.** Context window 3,072 tokens - longer states are truncated from the end, and
the question always keeps its reserve, so your options are never lost. Request payload
max 6 MB; invocation max 60 s. `choice` takes 2-255 options, `score` takes 2-10 levels.

**Text only.** The schema has an `images` field but this engine refuses it rather than
ignoring it.

**Determinism.** The same request gives the same answer, but probabilities can differ in
the last few digits depending on what else was batched alongside it (bf16 addition is not
associative, and batch shape changes the reduction order). Drift is ~0.003-0.007. Gate on
the **decision** - the argmax, the side of 0.5, the rounded level - not on an exact
probability.

**Cost.** The endpoint is one `ml.g6.xlarge` at **$1.1267/hr = ~$27/day**, billed whether
you use it or not. At saturation that works out to ~$0.022 per 1,000 requests; idle, it is
just $27/day. If you are the only user, consider tearing it down between sessions:

```bash
deploy/create_endpoint.py --name strands-decider-g6 --delete --region us-west-2
```

Recreating it takes about 12 minutes (the 20.6 GB image pull dominates), and the image
stays in ECR, so it is just the `create_endpoint.py` call again.